In [27]:
from WindPy import w
w.start()
from datetime import datetime
import ratespricer as rp
import pymysql

In [22]:
date_in=datetime.today()

In [11]:
data=w.wss("DR001.IB,DR007.IB,DR014.IB,R001.IB,R007.IB,R014.IB,FDR001.IR,FDR007.IR,FDR014.IR,FR001.IR,FR007.IR,FR014.IR", "vwap","tradeDate=20230714;cycle=D;priceAdj=U")

In [20]:
data

.ErrorCode=0
.Codes=[DR001.IB,DR007.IB,DR014.IB,R001.IB,R007.IB,R014.IB,FDR001.IR,FDR007.IR,FDR014.IR,FR001.IR,...]
.Fields=[VWAP]
.Times=[20230717 16:44:41]
.Data=[[1.3199,1.8154,1.9094,1.4629,1.9624,2.0531,1.34,1.8,1.9,1.5,...]]

In [18]:
repo_upload=pd.DataFrame(data.Data,columns=['DR001','DR007','DR014','R001','R007','R014','FDR001','FDR007','FDR014','FR001','FR007','FR014'])
repo_upload

,DR001,DR007,DR014,R001,R007,R014,FDR001,FDR007,FDR014,FR001,FR007,FR014
0,1.3199,1.8154,1.9094,1.4629,1.9624,2.0531,1.34,1.8,1.9,1.5,1.9,2.04


In [37]:
def u_bond_cnbd(date_input):
    if not w.isconnected():
        w.start()
    print('updating repo_vwap')
    di = rp.d_to_ymd(date_input)
    cnn = pymysql.connect(host='192.168.119.55', user='intern', passwd='rates2022C!', db='bond', charset='utf8')
    cur = cnn.cursor()
    replace_REPO = 'replace into bond.cfets_repo_rec_1d(DR001, DR007, DR014,R001,R007,R014,FDR001,FDR007,FDR014,FR001,FR007,FR014,dates) ' \
                       'values (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)'
    tmp =w.wss("DR001.IB,DR007.IB,DR014.IB,R001.IB,R007.IB,R014.IB,FDR001.IR,FDR007.IR,FDR014.IR,FR001.IR,FR007.IR,FR014.IR", "vwap","tradeDate="+date_input+";cycle=D;priceAdj=U") 
    repo_upload=pd.DataFrame(tmp.Data,columns=['DR001','DR007','DR014','R001','R007','R014','FDR001','FDR007','FDR014','FR001','FR007','FR014'])
    repo_upload['dates']=date_input
    print(repo_upload)
    cur.executemany(replace_REPO, repo_upload.values.tolist())
    cnn.commit()
    print('updating complete')
    cur.close()
    cnn.close()

In [38]:
date_input="2023-07-14"
u_bond_cnbd(date_input)

updating repo_vwap
    DR001   DR007   DR014    R001    R007    R014  FDR001  FDR007  FDR014  \
0  1.3199  1.8154  1.9094  1.4629  1.9624  2.0531    1.34     1.8     1.9   

   FR001  FR007  FR014       dates  
0    1.5    1.9   2.04  2023-07-14  
updating complete


In [48]:
df = pd.read_excel(r"C:\Users\Administrator.DESKTOP-S17J2N3\Desktop\回购利率历史.xlsx", header=0)
df = df.iloc[0:-3] # 去掉最后一行
df=df.fillna(0)
df

D:\2.app_install\Anaconda\lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,指标名称,DR001,DR007,DR014,R001,R007,R014,FDR001,FDR007,FDR014,FR001,FR007,FR014
0,2010-01-04,0.0000,0.0000,0.0000,1.1201,1.4034,1.7110,0.0000,0.00,0.0000,1.1250,1.4400,1.65
1,2010-01-05,0.0000,0.0000,0.0000,1.1186,1.3980,1.5426,0.0000,0.00,0.0000,1.1210,1.4000,1.60
2,2010-01-06,0.0000,0.0000,0.0000,1.1236,1.3899,1.5624,0.0000,0.00,0.0000,1.1313,1.4047,1.56
3,2010-01-07,0.0000,0.0000,0.0000,1.0831,1.3695,1.3609,0.0000,0.00,0.0000,1.0865,1.3936,1.36
4,2010-01-08,0.0000,0.0000,0.0000,1.0637,1.3898,1.3605,0.0000,0.00,0.0000,1.0690,1.3900,1.35
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3374,2023-07-10,1.1454,1.6978,1.7482,1.3024,1.8894,1.9410,1.1500,1.72,1.7300,1.2700,1.8600,1.80
3375,2023-07-11,1.1839,1.7270,1.7615,1.3333,1.9081,1.9706,1.1900,1.73,1.7535,1.3000,1.9000,1.90
3376,2023-07-12,1.2256,1.7526,1.7660,1.3710,1.9102,1.9751,1.2400,1.75,1.7500,1.3500,1.8500,1.95
3377,2023-07-13,1.4453,1.7851,1.8000,1.5772,1.9134,2.0046,1.4535,1.78,1.8000,1.5700,1.8500,2.00


In [49]:
cnn = pymysql.connect(host='192.168.119.55', user='intern', passwd='rates2022C!', db='bond', charset='utf8')
cur = cnn.cursor()
replace_REPO = 'replace into bond.cfets_repo_rec_1d(dates,DR001, DR007, DR014,R001,R007,R014,FDR001,FDR007,FDR014,FR001,FR007,FR014) ' \
                       'values (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)'
cur.executemany(replace_REPO, df.values.tolist())
cnn.commit()
cur.close()
cnn.close()